# EuRoC Spectrogram

Fit spectral or pseudo-spectral basis functions to fixed-duration intervals in a merged EuRoC CSV.

This notebook is intentionally thin: fitting, diagnostics, derived IMU signals, and Plotly figure builders live in `imuFactors.spectrogram`. Choose the GTSAM Chebyshev1 spectral basis (`gtsam.Chebyshev1Basis`), Chebyshev2 pseudo-spectral CGL node-value basis (`gtsam.Chebyshev2`), or Fourier basis (`gtsam.FourierBasis`). `N` is the number of fitted parameters. Chebyshev2 also supports a first-derivative Sobolev penalty through `lambda1`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any

from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception as exc:
    HAS_WIDGETS = False
    WIDGET_IMPORT_ERROR = exc

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python" / "imuFactors").exists():
    REPO_ROOT = Path("/Users/dellaert/git/imuFactors")

PYTHON_DIR = REPO_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

import imuFactors.spectrogram as spectrogram

DATA_DIR = REPO_ROOT / "data" / "euroc"
DATA_FILES = spectrogram.discover_euroc_files(DATA_DIR)
if not DATA_FILES:
    raise FileNotFoundError(f"No EuRoC CSV files found in {DATA_DIR}")

DEFAULT_FILE = DATA_DIR / "euroc_MH01.csv"
if not DEFAULT_FILE.exists():
    DEFAULT_FILE = DATA_FILES[0]

DEFAULT_N = 16
DEFAULT_WINDOW_SECONDS = spectrogram.WINDOW_SECONDS
DEFAULT_BASIS = spectrogram.DEFAULT_BASIS
DEFAULT_SIGNAL_GROUP = spectrogram.DEFAULT_SIGNAL_GROUP
DEFAULT_LAMBDA1 = 0.0
print(f"Found {len(DATA_FILES)} EuRoC CSV files in {DATA_DIR}")

In [ ]:
def run_dashboard(
    path: str | Path = DEFAULT_FILE,
    coefficient_count: int = DEFAULT_N,
    basis: str = DEFAULT_BASIS,
    signal_group: str = DEFAULT_SIGNAL_GROUP,
    window_seconds: float = DEFAULT_WINDOW_SECONDS,
    lambda1: float = DEFAULT_LAMBDA1,
    max_components: int = 6,
):
    """Fit a file and render the standard spectrogram views."""
    result = spectrogram.fit_spectral_windows(
        path,
        coefficient_count=int(coefficient_count),
        basis=basis,
        signal_group=signal_group,
        window_seconds=float(window_seconds),
        lambda1=float(lambda1),
    )
    selected = spectrogram.characteristic_windows(result)

    spectrogram.plot_average_spectra(result).show()
    spectrogram.plot_coefficient_spectrogram(result).show()
    spectrogram.plot_average_spectrogram_parts(result).show()
    
    display(spectrogram.interval_metrics_table(result, selected))
    spectrogram.plot_window_characteristics(result, selected).show()

    for label, window_index in selected.items():
        spectrogram.plot_interval_fit(
            result,
            window_index,
            label=label,
            max_components=max_components,
        ).show()
        spectrogram.plot_interval_coefficients(result, window_index, label=label).show()

    display(spectrogram.summary_table(result))
    return result

## Interactive Controls

Choose a merged EuRoC CSV, basis family, fitted parameter count `N`, interval length, signal group, and optional `lambda1` first-derivative penalty. Derived signal groups include `imu_norms` for `[gyro_norm, accel_norm]` and `accel_gravity_compensated` for accelerometer axes after subtracting orientation-predicted gravity. Fourier fits map each interval to one cycle on `[0, 2*pi]`; Chebyshev1 fits map each interval to `[-1, 1]`; Chebyshev2 fits pseudo-spectral CGL node values on `[0, 1]`. `lambda1` is only active for Chebyshev2.

In [ ]:
def make_dashboard_controls() -> tuple[Any, Any] | tuple[None, None]:
    if not HAS_WIDGETS:
        print("ipywidgets is not available; edit the DEFAULT_* values manually.")
        print(f"Widget import error: {WIDGET_IMPORT_ERROR!r}")
        return None, None

    file_dropdown = widgets.Dropdown(
        options=[(path.name, str(path)) for path in DATA_FILES],
        value=str(DEFAULT_FILE),
        description="file",
        layout=widgets.Layout(width="560px"),
    )
    basis_dropdown = widgets.Dropdown(
        options=list(spectrogram.BASIS_OPTIONS),
        value=DEFAULT_BASIS,
        description="basis",
        layout=widgets.Layout(width="560px"),
    )
    n_slider = widgets.IntSlider(
        value=DEFAULT_N,
        min=2,
        max=80,
        step=1,
        description="N",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    interval_slider = widgets.FloatSlider(
        value=DEFAULT_WINDOW_SECONDS,
        min=0.5,
        max=5.0,
        step=0.25,
        description="interval s",
        continuous_update=False,
        readout_format=".2f",
        layout=widgets.Layout(width="560px"),
    )
    lambda1_text = widgets.FloatText(
        value=DEFAULT_LAMBDA1,
        description="lambda1",
        continuous_update=False,
        layout=widgets.Layout(width="260px"),
    )
    group_dropdown = widgets.Dropdown(
        options=list(spectrogram.SIGNAL_GROUPS.keys()),
        value=DEFAULT_SIGNAL_GROUP,
        description="signals",
        layout=widgets.Layout(width="560px"),
    )
    component_slider = widgets.IntSlider(
        value=6,
        min=1,
        max=12,
        step=1,
        description="plot dims",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    run_button = widgets.Button(description="Run fit", button_style="primary", icon="play")
    output = widgets.Output()

    def sync_lambda1_enabled(*_: Any) -> None:
        lambda1_text.disabled = str(basis_dropdown.value) != "chebyshev2"

    basis_dropdown.observe(sync_lambda1_enabled, names="value")
    sync_lambda1_enabled()

    def on_run(_: Any) -> None:
        with output:
            clear_output(wait=True)
            global LAST_RESULT
            lambda1 = (
                float(lambda1_text.value)
                if str(basis_dropdown.value) == "chebyshev2"
                else 0.0
            )
            LAST_RESULT = run_dashboard(
                Path(file_dropdown.value),
                coefficient_count=int(n_slider.value),
                basis=str(basis_dropdown.value),
                signal_group=str(group_dropdown.value),
                window_seconds=float(interval_slider.value),
                lambda1=lambda1,
                max_components=int(component_slider.value),
            )

    run_button.on_click(on_run)
    controls = widgets.VBox([
        widgets.HBox([file_dropdown]),
        widgets.HBox([basis_dropdown]),
        widgets.HBox([n_slider]),
        widgets.HBox([interval_slider]),
        widgets.HBox([lambda1_text]),
        widgets.HBox([group_dropdown]),
        widgets.HBox([component_slider, run_button]),
    ])
    return controls, output


controls, output = make_dashboard_controls()
if controls is not None:
    display(controls, output)
    with output:
        LAST_RESULT = run_dashboard(DEFAULT_FILE, DEFAULT_N, DEFAULT_BASIS, DEFAULT_SIGNAL_GROUP)
else:
    LAST_RESULT = run_dashboard(DEFAULT_FILE, DEFAULT_N, DEFAULT_BASIS, DEFAULT_SIGNAL_GROUP)

## Working With Results

`LAST_RESULT.coeffs` has shape `(m, N, d)`: one row per interval, one fitted parameter per basis function or pseudo-spectral node, and one selected signal component. `LAST_RESULT.coeff_energy` is the standardized `m x N` image used for the spectrogram.

Raw coefficient heatmaps show the actual least-squares fitted parameters. Spectrogram and average spectra use robust component scaling so signals with different units can share one image. Derived auxiliary signals include `gyro_norm`, `accel_norm`, and gravity-compensated accelerometer columns `a_gc_x`, `a_gc_y`, `a_gc_z`. For Chebyshev2, positive `lambda1` adds a CGL Sobolev penalty on `integral |p'(t)|^2 dt`.